# Phishing URL Classifier Demo

This notebook is an early research demonstration for the proposed study: **AI-Assisted Mobile Phishing Awareness and Detection**.

The purpose is not to build a production security product. The purpose is to show a realistic research workflow: feature extraction, machine-learning classification, evaluation metrics, and translation of model output into mobile warning messages.

In [ ]:
import re
from urllib.parse import urlparse

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 1. Load sample data

The sample dataset is intentionally small. In the full study, this would be replaced with a larger public phishing URL dataset.

In [ ]:
df = pd.read_csv('../data/sample_urls.csv')
df.head()

## 2. URL feature extraction

The features below are simple and interpretable, which makes them suitable for a human-centred cybersecurity awareness study.

In [ ]:
def extract_features(url: str) -> dict:
    parsed = urlparse(url)
    hostname = parsed.netloc.lower()
    path = parsed.path.lower()
    suspicious_words = ['login', 'verify', 'update', 'secure', 'password', 'reset', 'urgent', 'claim', 'free']
    return {
        'url_length': len(url),
        'hostname_length': len(hostname),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'uses_https': 1 if parsed.scheme == 'https' else 0,
        'has_at_symbol': 1 if '@' in url else 0,
        'has_ip_address': 1 if re.search(r'\d+\.\d+\.\d+\.\d+', hostname) else 0,
        'suspicious_word_count': sum(word in url.lower() for word in suspicious_words),
        'path_length': len(path),
    }

features = pd.DataFrame([extract_features(url) for url in df['url']])
labels = df['label'].map({'legitimate': 0, 'phishing': 1})
features.head()

## 3. Train a simple classifier

A simple logistic regression model is used because it is interpretable and suitable for explaining risk indicators to non-expert users.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.33, random_state=42, stratify=labels
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)


## 4. Evaluate the model

The full research would report accuracy, precision, recall, F1-score, and confusion matrix using a larger validated dataset.

In [ ]:
results = {
    'accuracy': accuracy_score(y_test, predictions),
    'precision': precision_score(y_test, predictions, zero_division=0),
    'recall': recall_score(y_test, predictions, zero_division=0),
    'f1_score': f1_score(y_test, predictions, zero_division=0),
}

pd.DataFrame([results])

In [ ]:
print(classification_report(y_test, predictions, target_names=['legitimate', 'phishing'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions))

## 5. Convert model output into a mobile warning concept

The proposed research is human-centred. Therefore, the model result should be converted into a clear warning that helps users understand why a link may be risky.

In [ ]:
def warning_card(url: str) -> str:
    row = pd.DataFrame([extract_features(url)])
    risk = model.predict_proba(row)[0][1]
    if risk >= 0.70:
        level = 'High risk'
        message = 'This link shows multiple signs commonly associated with phishing. Check the sender, domain name, and login request before continuing.'
    elif risk >= 0.40:
        level = 'Medium risk'
        message = 'This link has some suspicious features. Review it carefully before entering personal information.'
    else:
        level = 'Low risk'
        message = 'No major suspicious patterns were detected, but you should still verify the sender and website.'
    return f'{level}: {message} Risk score: {risk:.2f}'

warning_card('http://paypal-security-update-login.verify-user-info.com')

## Research extension

The full Master by Research project can extend this demonstration by using a larger public dataset, comparing multiple machine-learning models, and running a controlled user study to test whether warning cards improve phishing awareness.